# Week 2: Build a Classification Model
Predicting Titanic passenger survival (0/1) using Logistic Regression and a Decision Tree

## 1. Load the Cleaned Dataset
Reusing the cleaned Titanic dataset from Week 1.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

df = pd.read_csv('https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv')

# Quick clean (same as Week 1)
df['age'] = df['age'].fillna(df['age'].median())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])
df['embark_town'] = df['embark_town'].fillna(df['embark_town'].mode()[0])
df['deck'] = df['deck'].fillna('Unknown')
df.drop_duplicates(inplace=True)

df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,Unknown,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,Unknown,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,Unknown,Southampton,no,True


## 2. Identify Target and Feature Columns
Our target is `survived` (0 = did not survive, 1 = survived) — a binary outcome.
We'll use passenger class, sex, age, family size, fare, and port of embarkation as features.

In [2]:
target = 'survived'
features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']

X = df[features]
y = df[target]

X.head()

,pclass,sex,age,sibsp,parch,fare,embarked
0,3,male,22.0,1,0,7.2500,S
1,1,female,38.0,1,0,71.2833,C
2,3,female,26.0,0,0,7.9250,S
3,1,female,35.0,1,0,53.1000,S
4,3,male,35.0,0,0,8.0500,S


## 3. Convert Categorical Columns to Numbers
`sex` and `embarked` are text categories. Models need numeric input, so we use
`pd.get_dummies()` to one-hot encode them.

In [3]:
X = pd.get_dummies(X, columns=['sex', 'embarked'], drop_first=True)
X.head()

,pclass,age,sibsp,parch,fare,sex_male,embarked_Q,embarked_S
0,3,22.0,1,0,7.2500,True,False,True
1,1,38.0,1,0,71.2833,False,False,False
2,3,26.0,0,0,7.9250,False,False,True
3,1,35.0,1,0,53.1000,False,False,True
4,3,35.0,0,0,8.0500,True,False,True


## 4. Split into Training and Testing Sets
80% of the data trains the model, 20% is held out to test it.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

(624, 8) (157, 8)


## 5. Train a Logistic Regression Model

In [5]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

## 6. Evaluate the Model
We use accuracy (overall correctness) plus precision, recall, and F1-score
(how well it identifies survivors specifically).

In [6]:
print("Accuracy:", accuracy_score(y_test, predictions))
print("Precision:", precision_score(y_test, predictions))
print("Recall:", recall_score(y_test, predictions))
print("F1-score:", f1_score(y_test, predictions))

Accuracy: 0.7961783439490446
Precision: 0.7903225806451613
Recall: 0.7205882352941176
F1-score: 0.7538461538461538


## 7. Try a Different Model: Decision Tree
Comparing against Logistic Regression to see if a different algorithm performs better.

In [7]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
dt_predictions = dt_model.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, dt_predictions))
print("Decision Tree F1-score:", f1_score(y_test, dt_predictions))

Decision Tree Accuracy: 0.7515923566878981
Decision Tree F1-score: 0.688


## 8. Summary and Interpretation

The Logistic Regression model achieved an accuracy of ~80% and an F1-score of
~0.75 on the test set, outperforming the Decision Tree (~75% accuracy, ~0.69 F1).
This suggests survival is reasonably well predicted by passenger class, sex, age,
and fare — consistent with the historical account that class and gender strongly
affected access to lifeboats. The model isn't perfect: it likely struggles with
edge cases where these features don't clearly separate survivors from
non-survivors (e.g. a wealthy older male, or a poor young woman). To improve it,
I'd try engineering new features (e.g. family size from `sibsp` + `parch`,
or title extracted from name), tuning hyperparameters, or trying an ensemble
model like Random Forest.